[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Ondas_y_Optica_%28OPT%29/Optica_%28physics.optics%29/OPT-06_Simulacion_Rayos_Agujero_Negro_Analogo.ipynb)

# [OPT-06] Óptica: Gravedad Analógica en Metamaterial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gravedad Analógica y Óptica de Transformación en Metamateriales
# Simulación de Trazado de Rayos Ópticos emulando la métrica de un Agujero Negro.
# El índice de refracción del metamaterial crece hacia el infinito en el centro.

# Espacio 2D
grid_size = 100
X, Y = np.meshgrid(np.linspace(-5, 5, grid_size), np.linspace(-5, 5, grid_size))
R = np.sqrt(X**2 + Y**2)

# Índice de refracción Analógico al Agujero Negro
# n(r) = 1 / (1 - r_schwarzschild / r). Cuando r -> r_s, n -> Infinito.
r_s = 1.0 # Radio de Schwarzschild análogo (El "Horizonte Óptico")
n_refract = np.ones_like(R)
valid = R > r_s
n_refract[valid] = 1.0 + r_s / (R[valid] - r_s)
n_refract[~valid] = np.nan # Centro devorador (Absorción infinita)

plt.figure(figsize=(10, 8))

# Fondo: Densidad del Índice de Refracción
plt.contourf(X, Y, n_refract, levels=50, cmap='inferno', vmin=1.0, vmax=5.0)
plt.colorbar(label="Índice de Refracción $n(r)$")
plt.scatter([0], [0], color='black', s=100, label="Singularidad Óptica")

# Horizonte de eventos
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(r_s*np.cos(theta), r_s*np.sin(theta), 'w--', lw=2, label="Horizonte Analógico")

# Trazado de Rayos Numérico Simplificado (Geodésicas en Óptica)
def trace_ray(y0, steps=200, dt=0.05):
    # Ecuación de rayos de la eikonal en gradiente de n
    x, y = -4.5, y0
    vx, vy = 1.0, 0.0 # Incide desde la izquierda horizontalmente
    
    path_x, path_y = [x], [y]
    
    for _ in range(steps):
        r_current = np.sqrt(x**2 + y**2)
        if r_current <= r_s + 0.1: break # Devorado por el "agujero"
        
        # Gradiente negativo atrayente hacia el alto índice de refracción
        # dn/dr ~ -1/(r-rs)^2
        grad_mag = -r_s / (r_current - r_s)**2
        ax = grad_mag * (x / r_current)
        ay = grad_mag * (y / r_current)
        
        vx += ax * dt
        vy += ay * dt
        # Normalizamos la velocidad de la luz en el medio
        v_mag = np.sqrt(vx**2 + vy**2)
        vx /= v_mag; vy /= v_mag
        
        x += vx * dt; y += vy * dt
        path_x.append(x); path_y.append(y)
        
    return path_x, path_y

# Lanzamos rayos de luz
for y_init in np.linspace(-3, 3, 11):
    px, py = trace_ray(y_init)
    plt.plot(px, py, 'cyan', lw=1.5)

plt.title("Gravedad Análoga: Agujero Negro en Metamaterial Óptico", fontsize=14)
plt.xlabel("X (Metamaterial)")
plt.ylabel("Y (Metamaterial)")
plt.legend()
plt.xlim(-5, 5)
plt.ylim(-5, 5)
plt.show()

# FÍSICA: Los rayos (luz láser en el metamaterial) sufren deflexión óptica idéntica 
# a las geodésicas del espacio-tiempo de Einstein. Los rayos cercanos al horizonte 
# quedan atrapados en órbitas ópticas infinitas o se precipitan hacia la singularidad.

